# Análisis de sentimiento
## Aplicación de un modelo entrenado para predecir en tiempo real

 
## Aprendizaje supervisado: un problema de clasificación

Los algoritmos de aprendizaje supervisado utilizan datos etiquetados en los que tanto la entrada como el resultado objetivo (*etiqueta*), se proporcionan al algoritmo. El aprendizaje supervisado también se denomina modelado predictivo o análisis predictivo, porque crea un modelo que es capaz de realizar predicciones.

Un algoritmo de clasificación toma un conjunto de datos con etiquetas conocidas y características predeterminadas, y aprende cómo etiquetar nuevos registros en función de esa información. Las características definen a cada individuo (cada registro, fila de nuestros datos, también llamado *ejemplo*). La etiqueta es la salida que corresponde con esas características. 

Veamos un ejemplo de clasificación de texto. 
* ¿Qué estamos tratando de predecir?
  * Si una revisión de producto es positiva o negativa.
  * Retrasada es la etiqueta: 1 para positivo 0 para negativo
* ¿Cuáles son las propiedades que puede utilizar para hacer predicciones?
  * Las palabras del texto de revisión se utilizan como características para descubrir similitudes y categorizar el sentimiento del texto del cliente como positivo o negativo.

### Regresión logística

La regresión logística es un método popular para predecir una respuesta binaria. Es un caso especial de modelos lineales generalizados que predice la probabilidad de que la clase asociada a un ejemplo sea una de las clases, o bien la otra (suele usarse casi siempre en problemas donde los registros pertenecen a una de entre dos clases posibles). La regresión logística mide la relación entre la "etiqueta" ***Y*** y las "características" ***X*** a través la estimación de probabilidades mediante una función logística. El modelo predice una probabilidad que se utiliza para predecir la clase a la que pertenece ese ejemplo.

### Dataset de opiniones de Amazon

Se puede descargar desde <a target="_blank" href="http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Sports_and_Outdoors_5.json.gz">aquí</a>

In [1]:
import os
from databricks.connect import DatabricksSession
from dotenv import load_dotenv

# Initialize Spark session and environment variables
spark = DatabricksSession.builder.getOrCreate()
load_dotenv()
STORAGE_ACCOUNT_NAME = os.getenv("STORAGE_ACCOUNT_NAME")
AZURE_ACCOUNT_KEY = os.getenv("AZURE_ACCOUNT_KEY")
EVENT_HUBS_ACCESS_KEY = os.getenv("EVENT_HUBS_ACCESS_KEY")
EVENT_HUBS_SERVICE = os.getenv("EVENT_HUBS_SERVICE")

#### Descargamos el dataset, lo movemos a DBFS y de ahí lo copiamos a ADLS

In [3]:
# !wget http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Sports_and_Outdoors_5.json.gz
# !gunzip reviews_Sports_and_Outdoors_5.json.gz

In [13]:
import urllib.request
import gzip
import shutil
from pathlib import Path

url = "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Sports_and_Outdoors_5.json.gz"
gz_file = "reviews_Sports_and_Outdoors_5.json.gz"
json_file = "reviews_Sports_and_Outdoors_5.json"

data_folder = Path("data")
data_folder.mkdir(exist_ok=True)

urllib.request.urlretrieve(url, data_folder / gz_file)

with gzip.open(data_folder / gz_file, 'rb') as gz_f:
    with open(data_folder / json_file, 'wb') as json_f:
        shutil.copyfileobj(gz_f, json_f)

In [0]:
# !mkdir /dbfs/FileStore
# !mv reviews_Sports_and_Outdoors_5.json /dbfs/FileStore/

In [2]:
catalog_name = "workspace"
schema_name = "amazon"
volume_name = "reviews"

In [16]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

DataFrame[]

In [18]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{json_file}"

with open(data_folder / json_file, "rb") as f:
    w.files.upload(volume_path, f, overwrite=True)

In [0]:
# !ls /dbfs/FileStore

In [19]:
# dbutils.fs.cp("dbfs:/FileStore/reviews_Sports_and_Outdoors_5.json", "abfss://datos@cursosparkucm.dfs.core.windows.net/")

In [3]:
ADLS_DESTINATION = f"abfss://datos@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/amazon-reviews"

In [26]:
df = spark.read.json(volume_path)
(df.write.option(f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", AZURE_ACCOUNT_KEY)
                .mode("overwrite")
                .json(ADLS_DESTINATION))

### Preprocesamiento y entrenamiento del modelo

Tenemos un conjunto de datos formado por textos breves escritos por clientes de Amazon al recibir su compra, en los cuales cada cliente expresa su opinión sobre el producto adquirido. Cada registro (cada fila del dataset) representa la opinión frente a algún producto. El texto tiene, por un lado, una columna en la que el cliente da un titular o resumen a su revisión, y por otro, una columna con un texto más largo donde expresa el detalle.

El dataset se encuentra en formato JSON-line en el que cada línea es un JSON completo, como el del siguiente ejemplo:

`{"reviewerID": "A1PUWI9RTQV19S", "asin": "B003Y5C132", "reviewerName": "kris", "helpful": [0, 1], "reviewText": "A little small in hind sight, but I did order a .30 cal box. Good condition, and keeps my ammo organized.", "overall": 5.0, "summary": "Nice ammo can", "unixReviewTime": 1384905600, "reviewTime": "11 20, 2013"}`

que, como vemos, sigue el siguiente esquema:

* **reviewerID** - identificador del cliente, p.ej. A2SUAM1J3GNN3B
* **asin** - identificador del producto, p.ej. 0000013714
* **reviewerName** - nombre del cliente
* **helpful** - valoración del grado de utilidad de esta opinión, expresado como un número real entre 0 y 1, p.ej. 2/3
* **reviewText** - texto de la opinión
* **overall** - valoración que da el cliente al producto, entre 1 y 5
* **summary** - resumen de la revisión
* **unixReviewTime** - instante en el que se creó esta opinión (expresado como unix time)
* **reviewTime** - instante en el que se creó esta opinión (formato en crudo)

In [4]:
from pyspark.sql import functions as F

rawDF = (spark.read
         .option(f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", AZURE_ACCOUNT_KEY)
         .option("inferSchema", "true")
         .json(ADLS_DESTINATION)
         )

# add column combining summary and review text, drop some others 
df = rawDF.withColumn("reviewTS",
                      F.concat_ws(" ", F.col("summary"), F.col("reviewText"))) \
    .drop("helpful", "reviewerID", "reviewerName", "reviewTime")

df.limit(5).show()

+----------+-------+--------------------+--------------------+--------------+--------------------+
|      asin|overall|          reviewText|             summary|unixReviewTime|            reviewTS|
+----------+-------+--------------------+--------------------+--------------+--------------------+
|B004V9598E|    4.0|Had to fiddle wit...|      Cheap Shootin'|    1367712000|Cheap Shootin' Ha...|
|B004V9598E|    4.0|I'm having a bit ...|       Pretty Badass|    1381449600|Pretty Badass I'm...|
|B004V9598E|    5.0|It needed a good ...|Best pellet rifle...|    1377388800|Best pellet rifle...|
|B004V9598E|    4.0|This is a great r...|Excellent pellet ...|    1344384000|Excellent pellet ...|
|B004V9598E|    5.0|I live in a devel...|           great gun|    1345680000|great gun I live ...|
+----------+-------+--------------------+--------------------+--------------+--------------------+



In [28]:
df.printSchema()

root
 |-- asin: string (nullable = true)
 |-- overall: double (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- unixReviewTime: long (nullable = true)
 |-- reviewTS: string (nullable = false)



Vamos a quitar las opiniones neutras (con valoración 3 sobre 5) para evitar posibles confusiones. Cualquier opinión con valoración 1 ó 2 será considerada negativa, mientras que una opinión con valoración 4 ó 5 será considerada positiva. Estas dos clases (negativa y positiva) serán los posibles valores de nuestra columna objetivo.

In [5]:
no_neutral_df = df.filter("overall !=3")
no_neutral_df.show()

+----------+-------+--------------------+--------------------+--------------+--------------------+
|      asin|overall|          reviewText|             summary|unixReviewTime|            reviewTS|
+----------+-------+--------------------+--------------------+--------------+--------------------+
|B004V9598E|    4.0|Had to fiddle wit...|      Cheap Shootin'|    1367712000|Cheap Shootin' Ha...|
|B004V9598E|    4.0|I'm having a bit ...|       Pretty Badass|    1381449600|Pretty Badass I'm...|
|B004V9598E|    5.0|It needed a good ...|Best pellet rifle...|    1377388800|Best pellet rifle...|
|B004V9598E|    4.0|This is a great r...|Excellent pellet ...|    1344384000|Excellent pellet ...|
|B004V9598E|    5.0|I live in a devel...|           great gun|    1345680000|great gun I live ...|
|B004V9598E|    5.0|this gun is great...|   this gun is great|    1370563200|this gun is great...|
|B004V9598E|    5.0|This rifle is the...|Having a great ti...|    1368403200|Having a great ti...|
|B004V9598

La función `describe()` nos da estadísticas de resumen acerca de una o varias columnas numéricas.

In [30]:
no_neutral_df.describe("overall").show()

+-------+-----------------+
|summary|          overall|
+-------+-----------------+
|  count|           272266|
|   mean| 4.51664548639933|
| stddev|0.934477779110063|
|    min|              1.0|
|    max|              5.0|
+-------+-----------------+



In [31]:
no_neutral_df.groupBy("overall").count().orderBy("overall").show()

+-------+------+
|overall| count|
+-------+------+
|    1.0|  9045|
|    2.0| 10204|
|    4.0| 64809|
|    5.0|188208|
+-------+------+



## Conversión de la valoración numérica en una etiqueta binaria

Vamos a crear, a partir de la columna `overall` que contiene la valoración numérica, una nueva columna binaria llamada `label` que será la que utilice nuestros algoritmo predictivo. Para ello utilizaremos un `Binarizer` de Spark, fijando el umbral en 3.0 (que nunca se da en nuestros datos porque ya lo habíamos quitado). Todo valor por debajo de este umbral será considerado como 0.0 y todo valor por encima será convertido en 1.0. La columna original `overall` no se modifica.

In [6]:
from pyspark.ml.feature import Binarizer
import numpy as np

binarizer = Binarizer(inputCol = "overall",
                      outputCol = "label",
                      threshold = 3.0)

binary_target_df = binarizer.transform(no_neutral_df)
binary_target_df.groupBy("overall","label").count().orderBy("overall").show()

+-------+-----+------+
|overall|label| count|
+-------+-----+------+
|    1.0|  0.0|  9045|
|    2.0|  0.0| 10204|
|    4.0|  1.0| 64809|
|    5.0|  1.0|188208|
+-------+-----+------+



## Muestreo estratificado
Como suele ser habitual en los problemas de clasificación binaria, existen muchos más ejemplos pertenecientes a una clase (en este caso la clase positiva) que a otra. Para que el modelo también sea sensible a ejemplos de la clase negativa, es conveniente tratar de equilibrar la proporción de ejemplos de cada clase presentes en nuestro conjunto de datos. Hay varias estrategias para conseguir esto. Aquí optamos por la más simple (y a la vez, la menos recomendable en problemas reales) que es eliminar ejemplos de la clase mayoritaria.

Utilizamos la función `sampleBy()` indicando la fracción de ejemplos de cada clase que queremos mantener. En este caso queremos mantener todos los ejemplos de la clase negativa (que son minoría), pero tan sólo queremos mantener el 10 % de los ejemplos de la clase mayoritaria. Si mostramos la cantidad de ejemplos en el DataFrame resultante de este muestreo, vemos que están más equilibrados aunque aún sigue ligeramente inclinado hacia la clase 1.0.

In [7]:
fractions = {1.0 : .1, 0.0 : 1.0}
balanced_df = binary_target_df.stat.sampleBy("label", fractions, 36)
balanced_df.groupBy("label").count().show()

+-----+-----+
|label|count|
+-----+-----+
|  1.0|25244|
|  0.0|19249|
+-----+-----+



Para poder saber cómo de bien funcionará el modelo entrenado en datos nuevos nunca vistos, vamos a dividir el conjunto de datos en subconjuntos de entrenamiento y de test. El conjunto de test se utilizará una sola vez, al final, cuando ya tengamos decidido y entrenado el modelo de predicción. El objetivo del conjunto de test será calcular una métrica que estime la bondad del modelo cuando sea puesto en producción y empiece a predecir datos sobre los que realmente no se conoce su etiqueta.

Usamos el 80 % de nuestros datos para entrenar, y el 20 % los dejamos fuera porque serán el conjunto de test.

In [8]:
split_seed = 5043
training_data, test_data = balanced_df.randomSplit([0.8, 0.2], split_seed)

# training_data.cache()
training_data.groupBy("label").count().show()

+-----+-----+
|label|count|
+-----+-----+
|  1.0|20216|
|  0.0|15444|
+-----+-----+



## Ingeniería de variables y pipelines

Para que las características sean utilizadas por un algoritmo de aprendizaje automático, deben transformarse y colocarse en vectores de características, que son vectores numéricos que representa el valor de cada característica. Los textos en sí mismos no son utilizables por los algoritmos hasta que no pasen a través de dicho proceso.

Spark ML proporciona un conjunto uniforme de API de alto nivel creadas sobre DataFrames. Usaremos un ML Pipeline para pasar los datos a través de transformadores con el fin de extraer las características y un estimador para producir el modelo.

* Transformador: Un transformador es un algoritmo que transforma un DataFrame en otro DataFrame. Usaremos un transformador para obtener un DataFrame con una columna de vector de características.

* Estimador: un estimador es un algoritmo que se puede ajustar a un DataFrame para producir un transformador. Usaremos un estimador que consistirá en un algoritmo de Regresión logística para entrenar un modelo. El modelo entrenado obtenido será un transformador que es capaz de transformar datos sobre los que no se conoce su etiqueta, para calcular predicciones.

* Pipeline: un pipeline encadena varios transformadores y estimadores para especificar un flujo de trabajo de aprendizaje automático. Usaremos un Pipeline de Spark ML para tener en una sola pieza toda la secuencia de transformaciones necesarias para preparar los datos hasta llegar al modelo. De esa manera, podemos entrenar la pieza (el pipeline) como un todo, y utilizarlo para que los datos nuevos también pasen a través de las mismas etapas que habíamos utilizado para preparar los datos de entrenamiento. Esto permite pre-procesar datos nuevos de la misma manera que se hizo con los datos de entrenamiento, siguiendo exactamente los mismos pasos.

Por último utilizaremos un evaluador para medir la bondad del modelo entrenado.

Empezaremos con las siguientes etapas de ingeniería de variables:

* Primero utilizamos un `RegexTokenizer` para separar cada texto en palabras. Esto transforma cada texto en un vector de strings con las palabras. Para más detalles: http://spark.apache.org/docs/latest/ml-features.html#tokenizer
* Después aplicaremos un `StopWordsRemover` para eliminar de cada vector de palabras aquellas sin significado, como artículos, preposiciones, etc. Para más detalles: http://spark.apache.org/docs/latest/ml-features.html#stopwordsremover


In [9]:
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover

spark = DatabricksSession.builder.getOrCreate()

tokenizer = RegexTokenizer(inputCol = "reviewTS",
                           outputCol = "reviewTokensUf",
                           pattern = "\\s+|[,.()\"]")

remover = StopWordsRemover(inputCol = "reviewTokensUf",
                           outputCol = "reviewTokens"
                          ).setStopWords(StopWordsRemover.loadDefaultStopWords("english"))

Utilizaremos el siguiente método para convertir vectores de palabras de un texto en vectores numéricos, utilizables por un algoritmo predictivo.

**De la documentación oficial de Spark**:

TF-IDF es un método de vectorización de características ampliamente utilizado en la minería de texto para reflejar la importancia de un término para un documento en el corpus. Si denotamos a un término (palabra) como *t*, un documento como *d* y el corpus como *D*, entonces:

* La Frecuencia de un término **TF(*t*, *d*)** es el número de veces que el término *t* aparece en el documento *d* 
* La frecuencia en el documento **DF(*t*, *D*)** es el número de documentos que contienen el término *t*.

Si solo usamos la frecuencia de los términos para medir la importancia, es muy fácil sobreenfatizar erróneamente los términos que aparecen con mucha frecuencia pero que contienen poca información sobre el documento, p. Ej. la palabra *fútbol* en un corpus compuesto por biografías de futbolistas. Si un término aparece con mucha frecuencia en el corpus, significa que no contiene información especial sobre un documento en particular. 

* La *frecuencia inversa de los documentos* **IDF(*t*, *D*)** es una medida numérica de cuánta información proporciona un término:

$$ IDF (t, D) = \frac{log | D | +1} {DF (t, D) +1} $$

donde | D | es el número total de documentos del corpus. Dado que se usa logaritmo, si un término aparece en todos los documentos, su valor IDF se convierte en 0. Tenga en cuenta que se aplica un término de suavizado para evitar dividir por cero para los términos fuera del corpus.

La medida TF-IDF es simplemente el producto de TF e IDF:
$$ TFIDF (t, d, D) = TF (t, d) ⋅ IDF (t, D) $$

Hay varias variantes en la definición de TF y de IDF. En Spark ML, están separados para que sean flexibles y poder combinarlos de varias maneras.

* `CountVectorizer()` cuenta las ocurrencias totales de cada palabra en todo el corpus de textos, y se queda con las N más relevantes, siendo N un parámetro especificado por el usuario (en nuestro caso, N = 20000). Tras esto, en cada texto contará el número de apariciones de cada una de esas N palabras seleccionadas. Por tanto, cada texto vendrá representado por un vector numérico de longitud 20000, y nuestro problema tendrá 20000 variables.

* `HashingTF()` es similar, pero cada posición no se asocia a una sola palabra sino que puede estar compartida por más de una. El usuario especifica también la dimensión N de los vectores obtenidos (se recomienda que sea una potencia de 2 debido a cómo actúa esta técnica). A grandes rasgos, cada palabra se codifica mediante un código que a su vez va a parar a una posición determinada del vector numérico que va a representar a ese texto, por lo que puede haber colisiones en algunas ocasiones, y que una posición sea utilizada para acumular las apariciones de más de una palabra diferente.

Se puede utilizar cualquiera de estas opciones, aunque no las dos simultáneamente.

In [10]:
from pyspark.ml.feature import CountVectorizer, IDF, HashingTF

count_vectorizer = CountVectorizer().setInputCol("reviewTokens")\
                                    .setOutputCol("cv")\
                                    .setVocabSize(2000).setMinDF(4)

idf = IDF().setInputCol("cv").setOutputCol("features")

In [11]:
from pyspark.ml.classification import LogisticRegression

# El último elemento del pipeline será un estimador, concretamente de la clase LogisticRegression
logisticRegression = LogisticRegression(featuresCol="features", labelCol="label")\
                                 .setMaxIter(100)\
                                 .setRegParam(0.02)\
                                 .setElasticNetParam(0.3)

## Configuramos el pipeline y lo entrenamos, como una sola cosa

In [12]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages = [tokenizer, remover, count_vectorizer, idf, logisticRegression])
pipelineModel = pipeline.fit(training_data)

#### Comprobamos qué palabras influyen más en el sentimiento

Examinamos los coeficientes de la regresión logística buscando los que son mayores en valor absoluto. Puesto que los datos de entrada son valores de TF-IDF, sus rangos son aproximadamente similares.

In [13]:
import pandas as pd

vocabulary = pipelineModel.stages[2].vocabulary

# De la lista de stages, nos quedamos con el último elemento (LogisticRegressionModel, ya entrenado: transformador)
lrModel = pipelineModel.stages[-1]

# Obtenemos el array de coeficientes, que vienen en el mismo orden que las variables
weights = lrModel.coefficients

# Lista de pares (palabra, coeficiente) 
word_weight = list(zip(vocabulary,weights))

word_weight.sort(key = lambda pair: np.abs(pair[1]), reverse = True)

# Convertimos la lista de pares en un DataFrame para poder imprimirlo de manera más clara
word_weight_df = pd.DataFrame(word_weight, columns = ["word", "weight"])[0:20]
word_weight_df

,word,weight
0,great,0.596867
1,returned,-0.366947
2,poor,-0.339896
3,perfect,0.305181
4,waste,-0.302215
5,broke,-0.278016
6,useless,-0.270633
7,disappointing,-0.263377
8,return,-0.262333
9,junk,-0.247200


In [29]:
import posixpath
original_join = os.path.join
os.path.join = posixpath.join

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.amazon.models")
model_path = f"/Volumes/{catalog_name}/{schema_name}/models/sentiment_pipeline"
pipelineModel.write().overwrite().save(model_path)

In [5]:
import posixpath
from pyspark.ml import PipelineModel

original_join = os.path.join
os.path.join = posixpath.join
model_path = f"/Volumes/{catalog_name}/{schema_name}/models/sentiment_pipeline"
pipelineModel = PipelineModel.load(model_path)

os.path.join = original_join

# Predicciones en tiempo real con el modelo entrenado

Vamos a usar Spark Structured Streaming para hacer predicciones sobre la partición de test. Vamos a escribir periódicamente en Kafka, en el topic `revisiones`, subconjuntos de opiniones, como si hubiera un productor real al otro lado, y Spark será notificado en tiempo real e irá leyendo los datos nuevos en un nuevo batch.

No necesitamos ninguna columna de `label` ni valoración ni similar, puesto que son datos para predecir, pero como venían en el dato original tal cual, no los hemos modificado.

#### Creación de los datos de Kafka
* Creamos el diccionario de propiedades con los valores adecuados que hemos obtenido de EventHubs.
* Creamos el cliente con el paquete Confluent Kafka a partir de dicho diccionario, para poder manejar nuestro cluster de Kafka programáticamente.
  * En realidad estamos manejando el servicio de Azure EventHubs "como si fuese" Kafka, aunque no tenemos como tal un cluster de Kafka.
* Creamos el topic en Kafka programáticamente (**solo debemos hacerlo la primera vez**)
* Leemos el fichero `/dbfs/FileStore/reviews_Sports_and_Outdoors_5.json` como lista de líneas en Python.
* Insertamos los datos poco a poco en lotes, simulando un productor externo que está continuamente escribiendo. 
  * Esto lo haremos en una celda posterior, cuando ya esté arrancado el flujo de lectura, para ir viendo cómo se crean distintos batches cuando Spark lee un batch de Kafka con todo lo que hay en ese momento, calcula predicciones y entonces vuelve a leer.

## Pega en la siguiente celda la Cadena de conexión que has copiado desde la configuración de Event Hubs

In [53]:
# Leemos el fichero de Python como líneas y las vamos insertando en Kafka
from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic
import socket

# PEGAMOS AQUÍ LA CADENA DE CONEXIÓN QUE HABÍAMOS COPIADO PROVISIONALMENTE EN EL BLOC DE NOTAS.
# SU ASPECTO DEBE SER SIMILAR A Endpoint=sb://nombre001eh.servicebus.windows.net/;SharedAccessKeyName....
connection_string = EVENT_HUBS_ACCESS_KEY

# eventhubs_service = EVENT_HUBS_SERVICE   # REEMPLAZA ESTO POR EL NOMBRE DE TU SERVICIO DE EVENT HUBS

conf = {"bootstrap.servers": f"{EVENT_HUBS_SERVICE}.servicebus.windows.net:9093",
        "sasl.mechanism": "PLAIN",
        "security.protocol": "SASL_SSL",
        "sasl.username": "$ConnectionString",
        "sasl.password": connection_string,
        "request.timeout.ms": "60000"
    }

admin_client = AdminClient(conf)

#### Creación del topic `revisiones` si no existiera

In [7]:
revisiones_topic = NewTopic("revisiones", num_partitions=3, replication_factor=3)
fs = admin_client.create_topics([revisiones_topic])

# Esperamos a que termine la operación
for topic, f in fs.items():
    try:
        f.result()  # The result itself is None
        print(f"Topic {topic} creado")
    except Exception as e:
        print(f"Fallo al crear el topic {topic}: {e}")

Topic revisiones creado


#### Borrado del topic, si quisiéramos "vaciarlo" para luego crearlo de nuevo

#### La siguiente celda no debe ser ejecutada si todo está yendo bien

In [54]:
# Borrado del topic si fuera necesario

fs = admin_client.delete_topics(["revisiones"])
# Esperar a que termine la operación
for topic, f in fs.items():
    try:
        f.result()  # The result itself is None
        print("Topic {} borrado".format(topic))
    except Exception as e:
        print("Failed to delete topic {}: {}".format(topic, e))

Topic revisiones borrado


### Lectura desde Kafka

Vamos a dar cada dato como si fuera un JSON completo en una sola línea. Después despiezamos cada JSON (cada línea) para pasarlo a un DataFrame de una columna de tipo string. Leermos cada JSON como una única línea de tipo string obtenida de Apache Kafka, configurando las siguientes opciones:

  * Usamos la variable `readStream` (en lugar de `read` como solemos hacer) interna de la SparkSession `spark`
  * Indicamos que el formato es `"kafka"` con `.format("kafka")`
  * Indicamos cuáles son los brokers de Kafka de los que vamos a leer y el puerto al que queremos conectarnos para leer (9092 es el que usa Kafka por defecto), con `.option("kafka.bootstrap.servers", "broker0:9092,broker1:9092")`.
  * Indicamos que queremos subscribirnos al topic `"revisiones"` con `.option("subscribe", "revisiones")`.
  * Finalmente ponemos `load()` para realizar la lectura.

In [8]:
# Leemos de Kafka suscribiéndonos al topic "revisiones" (revisiones de productos)
username = conf["sasl.username"]
password = conf["sasl.password"]

textosStreamingDF = spark.readStream\
  .format("kafka")\
  .option("kafka.bootstrap.servers", conf["bootstrap.servers"])\
  .option("subscribe", "revisiones")\
  .option("kafka.security.protocol","SASL_SSL") \
  .option("kafka.sasl.mechanism", "PLAIN") \
  .option("kafka.sasl.jaas.config", 
          f"""kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{username}" password="{password}";""")\
  .load()

In [9]:
textosStreamingDF.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [13]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, ArrayType
from pyspark.sql import functions as F

# Esquema de cada mensaje
schema = StructType([
    StructField("asin", StringType(), True),
    StructField("helpful", ArrayType(LongType()), True),
    StructField("overall", StringType(), True),
    StructField("reviewText", StringType(), True),
    StructField("reviewTime", StringType(), True),
    StructField("reviewerName", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("unixReviewTime", LongType(), True)
])

parsedDF = (textosStreamingDF
    .withColumn("value", F.col("value").cast(StringType()))
    .withColumn("reviewStruct", F.from_json(F.col("value"), schema))
    .select("reviewStruct.*") # Nos permite recuperar los campos anidados como columnas
    .withColumn("reviewTS", F.concat(F.col("summary"), F.lit(" "), F.col("reviewText"))))

predictionsStreamingDF = pipelineModel.transform(parsedDF)\
    .withColumn("current_timestamp", F.current_timestamp())

In [14]:
parsedDF.printSchema()

root
 |-- asin: string (nullable = true)
 |-- helpful: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- overall: string (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- reviewTime: string (nullable = true)
 |-- reviewerName: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- unixReviewTime: long (nullable = true)
 |-- reviewTS: string (nullable = true)



In [15]:
predictionsStreamingDF.printSchema()

root
 |-- asin: string (nullable = true)
 |-- helpful: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- overall: string (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- reviewTime: string (nullable = true)
 |-- reviewerName: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- unixReviewTime: long (nullable = true)
 |-- reviewTS: string (nullable = true)
 |-- reviewTokensUf: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- reviewTokens: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- cv: vectorudt (nullable = true)
 |-- features: vectorudt (nullable = true)
 |-- rawPrediction: vectorudt (nullable = true)
 |-- probability: vectorudt (nullable = true)
 |-- prediction: double (nullable = false)
 |-- current_timestamp: timestamp (nullable = false)



In [0]:
#!ls -l /dbfs/FileStore/_checkpoints

In [0]:
! rm -r /dbfs/FileStore/_checkpoints

In [30]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
checkpoint_path = f"/Volumes/{catalog_name}/{schema_name}/reviews/_checkpoints"

In [31]:
try:
    w.dbutils.fs.rm(checkpoint_path, True)
    print("Checkpoint bookmark deleted! Stream will start from scratch.")

except Exception as e:
    print("No checkpoint found to delete (this is fine if it's your first run).")

Checkpoint bookmark deleted! Stream will start from scratch.


In [51]:
output = (predictionsStreamingDF
                    .writeStream
                    .queryName("predicciones")
                    .trigger(availableNow=True)
                    .outputMode("append")
                    .format("delta")
                    .option("checkpointLocation", checkpoint_path)
                    .toTable("workspace.amazon.predicciones")
          )

In [52]:
output.stop()

### Insertamos en Kafka los mismos datos con los que entrenamos
* Lo hacemos en bloques de 30.000 revisiones cada uno.

In [32]:
from confluent_kafka import Producer
import time
from typing import List

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
volume_path = "/Volumes/workspace/amazon/reviews/reviews_Sports_and_Outdoors_5.json"

with w.files.download(volume_path).contents as f:
    raw_lineas = f.read().splitlines()
    lista_lineas = [x.decode("utf-8").replace("\n", "") for x in raw_lineas]

producer = Producer(conf)

def escribe_lista(producer: Producer, topic: str, lista, tam_bloque=30000):
    i = 0
    finalizar = False
    while not finalizar:
        fin = min((i+1)*tam_bloque, len(lista))
        finalizar = (fin == len(lista))
        print(f"Enviando mensajes {i*tam_bloque} a {fin}")
        bloque = lista[(i*tam_bloque):fin]
        for mensaje in bloque:
            producer.produce(topic, value=mensaje)
        producer.flush()
        i = i+1

escribe_lista(producer, "revisiones", lista_lineas)


Enviando mensajes 0 a 30000
Enviando mensajes 30000 a 60000
Enviando mensajes 60000 a 90000
Enviando mensajes 90000 a 120000
Enviando mensajes 120000 a 150000
Enviando mensajes 150000 a 180000
Enviando mensajes 180000 a 210000
Enviando mensajes 210000 a 240000
Enviando mensajes 240000 a 270000
Enviando mensajes 270000 a 296337


In [41]:
spark.read.table("workspace.amazon.predicciones").limit(10).show()

+----------+-------+-------+--------------------+-----------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+
|      asin|helpful|overall|          reviewText| reviewTime|        reviewerName|             summary|unixReviewTime|            reviewTS|      reviewTokensUf|        reviewTokens|                  cv|            features|       rawPrediction|         probability|prediction|   current_timestamp|
+----------+-------+-------+--------------------+-----------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+
|1881509818| [0, 0]|    5.0|This came in on t...|01 26, 2014|        David Briner|      Woks very good|   

In [48]:
spark.read.table("workspace.amazon.predicciones")\
          .groupBy("current_timestamp").count()\
          .sort("current_timestamp").show()

+--------------------+------+
|   current_timestamp| count|
+--------------------+------+
|2026-03-06 03:12:...|296337|
+--------------------+------+



#### Vamos a insertar ahora 10000 mensajes para leerlos en modo batch

* Kafka continúa por donde se quedó y solamente leerá estos 1000 mensajes nuevos, porque estamos usando la misma carpeta de checkpoints existentes, donde estaban almacenados los offsets de la ejecución anterior.
* Para tener un control de grano fino, se pueden utilizar distintas ubicaciones de checkpoints, distintas para cada query.

In [44]:
escribe_lista(producer, "revisiones", lista_lineas[10000:20000], 10000)

Enviando mensajes 0 a 10000


In [46]:
output_batch = predictionsStreamingDF\
                    .writeStream\
                    .queryName("predicciones")\
                    .trigger(availableNow=True)\
                    .outputMode("append")\
                    .format("delta")\
                    .option("checkpointLocation", checkpoint_path)\
                    .toTable("amazon.reviews.predicciones")

In [47]:
try:
    w.dbutils.fs.rm(checkpoint_path, True)
    print("Checkpoint bookmark deleted! Stream will start from scratch.")

except Exception as e:
    print("No checkpoint found to delete (this is fine if it's your first run).")

Checkpoint bookmark deleted! Stream will start from scratch.
